# Generate an image batch from a prompt

Choose **SDXL Turbo** or **SDXL Base**, enter a prompt, set the image count and seed, then click **Generate batch**. Images and a `settings.json` file are saved under `outputs/interactive_generation/`.

Run this notebook from the debiasing repository on a CUDA GPU. The first use of a model downloads its weights from Hugging Face into the normal model cache; you may need to accept the model license on Hugging Face first. Install `ipywidgets` in the notebook environment if the controls below do not render.

In [ ]:
from pathlib import Path
import sys

# Find the repo root even if Jupyter started in a child directory.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "scripts/generate/generation_utils.py").exists())
sys.path.insert(0, str(ROOT))

import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
from scripts.generate.generation_utils import load_pipeline, generate_batch, model_defaults

if not torch.cuda.is_available():
    raise RuntimeError("This notebook currently needs a CUDA GPU. Start Jupyter on a GPU node.")

pipe = None
loaded_model = None


## Set up the batch

Turbo is run with its one-step, zero-guidance profile. Base uses the selected step count and guidance scale. Each image uses a different seed, starting at the seed below.

In [ ]:
model = widgets.Dropdown(options=["SDXL Turbo", "SDXL Base"], value="SDXL Base", description="Model:")
prompt = widgets.Textarea(value="A photo of a flower in a garden", description="Prompt:", layout=widgets.Layout(width="90%", height="90px"))
count = widgets.IntSlider(value=4, min=1, max=12, step=1, description="Images:")
seed = widgets.IntText(value=0, description="First seed:")
steps = widgets.IntSlider(value=30, min=1, max=50, step=1, description="Steps:")
guidance = widgets.FloatSlider(value=5.0, min=0.0, max=12.0, step=0.5, description="Guidance:")
button = widgets.Button(description="Generate batch", button_style="primary", icon="image")
output = widgets.Output()


def update_profile(change=None):
    defaults = model_defaults(model.value)
    turbo = model.value == "SDXL Turbo"
    steps.value = defaults["steps"]
    guidance.value = defaults["guidance"]
    steps.disabled = turbo
    guidance.disabled = turbo

model.observe(update_profile, names="value")
update_profile()

def on_generate(_):
    global pipe, loaded_model
    with output:
        clear_output(wait=True)
        try:
            if pipe is None or loaded_model != model.value:
                if pipe is not None:
                    del pipe
                    torch.cuda.empty_cache()
                print(f"Loading {model.value}…")
                pipe = load_pipeline(model.value)
                loaded_model = model.value
            print(f"Generating {count.value} images…")
            images, run_dir, settings = generate_batch(
                pipe=pipe,
                model_name=model.value,
                prompt=prompt.value,
                count=count.value,
                seed=seed.value,
                steps=steps.value,
                guidance=guidance.value,
                output_root=ROOT / "outputs/interactive_generation",
                run_name="prompt_batch",
            )
            print(f"Saved to {run_dir}")
            display(*images)
        except Exception as exc:
            print(f"Generation failed: {type(exc).__name__}: {exc}")

button.on_click(on_generate)
display(model, prompt, count, seed, steps, guidance, button, output)


### Model notes

- **SDXL Base**: defaults to 30 steps, guidance 5, and 1024 × 1024 output.
- **SDXL Turbo**: uses one step, guidance 0, and 512 × 512 output.
- Images are generated one at a time to keep GPU memory use lower. The saved settings record the model, prompt, seeds, steps, guidance, and image size.